In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))
print("Using project root:", project_root)

Using project root: /home/mbz/Documents/PhD/SCD41_DATA


In [2]:
from src.config import PipelineConfig
from src.pipeline import SensorPipeline
from src.logging_utils import setup_logging
config = PipelineConfig.from_yaml(Path("../config/sensors.yaml"))
print(f"Loaded {len(config.sensors)} sensors:")
for s in config.sensors:
    print(f"  - {s.name} ({s.sensor_type}, altitude: {s.altitude_m}m)")
setup_logging(config.log_dir, log_name="notebook_run")

pipeline = SensorPipeline(config, force=True)

2025-12-10 02:59:55,989 - root - INFO - Logging initialized. Log file: /home/mbz/Documents/PhD/SCD41_DATA/output/logs/notebook_run_20251210_025955.log


Loaded 5 sensors:
  - CO2_SCT1_2M (scd41, altitude: 2m)
  - CO2_30M_A (scd41, altitude: 30m)
  - CO2_30M_B (scd41, altitude: 30m)
  - CO2_30M_C (scd41, altitude: 30m)
  - LGR_30M (lgr_csv, altitude: 30m)


In [3]:
pipeline.run_all(year=2024, month=9)

2025-12-10 03:00:06,605 - src.pipeline - INFO - Starting pipeline for all sensors
2025-12-10 03:00:06,606 - src.pipeline - INFO - ============================================================
2025-12-10 03:00:06,606 - src.pipeline - INFO - Processing sensor: CO2_SCT1_2M
2025-12-10 03:00:06,606 - src.pipeline - INFO - ============================================================
2025-12-10 03:00:06,700 - src.pipeline - INFO - Found 19 days ((2024, 9, 12) to (2024, 9, 30))
2025-12-10 03:00:06,700 - src.processor - INFO - Processing 2024-09-12 (13 files)
2025-12-10 03:00:09,189 - src.processor - INFO - Removed 31389 duplicate timestamps
2025-12-10 03:00:09,193 - src.uptime - WARNING - Unexpected sample interval: 2.0s (expected ~60s)
2025-12-10 03:00:09,218 - src.processor - INFO - ✓ 2024-09-12: 16068 raw → 67 10-min samples (uptime: 100.0%)
2025-12-10 03:00:09,222 - src.processor - INFO - Processing 2024-09-13 (24 files)
2025-12-10 03:00:14,554 - src.processor - INFO - Removed 68209 duplica

In [ ]:
pipeline.run_all(year=2024, month=10)

In [ ]:
# from src.taylor_diagram_custom import TaylorPlotter

# plotter = TaylorPlotter(output_dir=Path("output/plots"))

# # Simple: all sensors
# fig = plotter.plot_from_config(
#     config,
#     title="Taylor Diagram: SCD41 Sensors vs LGR Reference",
#     filename="taylor_all_sensors.png"
# )

# # Selective: only 30m sensors
# fig = plotter.plot_from_config(
#     config,
#     test_sensor_ids=['CO2_30M_A', 'CO2_30M_B', 'CO2_30M_C'],
#     title="Taylor Diagram: 30m Height Sensors",
#     filename="taylor_30m_only.png"
# )

In [ ]:
from src.visualization import TaylorDiagram

taylor = TaylorDiagram(config.output_dir / "plots")
taylor.plot_from_config(config, normalize=True)

In [ ]:
from src.visualization import generate_all_plots

# Generate all standard plots
generate_all_plots(config)

In [4]:
from src.merger import merge_all_sensors

merged = merge_all_sensors(
    individual_dir=config.output_dir,
    sensor_names=[s.name for s in config.sensors]
)

print(f"Merged dataset shape: {merged.shape}")
print(f"Date range: {merged.index.min()} to {merged.index.max()}")
print(f"\nColumns: {list(merged.columns)}")

2025-12-10 03:06:03,230 - src.merger - INFO - Merging 5 sensors...
2025-12-10 03:06:03,231 - src.merger - INFO -   Loading CO2_SCT1_2M...
2025-12-10 03:06:03,241 - src.merger - INFO -     2449 rows (2024-09-12 12:20:00 to 2024-09-30 23:50:00)
2025-12-10 03:06:03,242 - src.merger - INFO -   Loading CO2_30M_A...
2025-12-10 03:06:03,245 - src.merger - INFO -     174 rows (2024-09-19 11:10:00 to 2024-09-28 11:00:00)
2025-12-10 03:06:03,246 - src.merger - INFO -   Loading CO2_30M_B...
2025-12-10 03:06:03,249 - src.merger - INFO -     373 rows (2024-09-19 11:10:00 to 2024-09-28 18:10:00)
2025-12-10 03:06:03,250 - src.merger - INFO -   Loading CO2_30M_C...
2025-12-10 03:06:03,259 - src.merger - WARNING -     ⚠️  Found 29 duplicate timestamps in CO2_30M_C
2025-12-10 03:06:03,259 - src.merger - INFO -     Keeping last occurrence of duplicates
2025-12-10 03:06:03,260 - src.merger - INFO -     4261 rows (2024-10-02 09:30:00 to 2024-10-31 23:30:00)
2025-12-10 03:06:03,261 - src.merger - INFO -   L

TypeError: '<' not supported between instances of 'str' and 'Timestamp'